# Model Error Analysis

This notebook analyzes model mistakes on the fake-news classification task. It reports confusion matrices, classification metrics, false positives, false negatives, and the relationship between text length and prediction errors.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch_geometric.loader import DataLoader

sys.path.insert(0, os.path.abspath('..'))

from src.data.dataset_builder import load_raw_parquet_data, create_pyg_graphs_from_raw, split_datasets_strict
from src.data.augment_prune import augment_dataset, prune_dataset
from src.models.hyd_fake import HyDFakeModel

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'font.size': 11,
    'savefig.dpi': 150,
    'figure.dpi': 150,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


def load_raw_text(dataset: str, raw_dir: str = '../data/raw') -> pd.DataFrame:
    raw_path = Path(raw_dir) / dataset / 'raw_text'
    parquet_files = sorted(raw_path.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found in {raw_path}')
    df = pd.read_parquet(parquet_files[0]).copy()
    text_col = 'text' if 'text' in df.columns else df.columns[0]
    df[text_col] = df[text_col].astype(str)
    df['text_length'] = df[text_col].str.split().str.len().fillna(0).astype(int)
    df.attrs['text_col'] = text_col
    return df


def generate_predictions_for_dataset(dataset: str, raw_dir: str = '../data/raw', interim_dir: str = '../data/interim', output_dir: str = '../outputs/hyd_fake') -> pd.DataFrame:
    model_path = Path(output_dir) / dataset / 'model_hyd_fake_final.pt'
    if not model_path.exists():
        raise FileNotFoundError(f'Model checkpoint not found: {model_path}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_dict = load_raw_parquet_data(dataset, raw_dir=raw_dir, interim_dir=interim_dir, encoder='sbert')
    graphs = augment_dataset(create_pyg_graphs_from_raw(data_dict))
    if dataset == 'gossipcop':
        graphs = prune_dataset(graphs, max_depth=3)

    _, _, test_graphs = split_datasets_strict(graphs, dataset_name=dataset, raw_dir=raw_dir)
    if not test_graphs:
        raise ValueError(f'No test graphs found for dataset: {dataset}')

    loader = DataLoader(test_graphs, batch_size=16, shuffle=False)
    model = HyDFakeModel(
        text_frozen_dim=test_graphs[0].text_x.shape[1],
        graph_in_dim=test_graphs[0].x.shape[1],
        hidden_dim=256,
        num_gat_layers=4,
        edge_prune_threshold=0.05,
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device).eval()

    raw_df = load_raw_text(dataset, raw_dir=raw_dir)
    text_col = raw_df.attrs['text_col']

    rows = []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.to(device))
            probabilities = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            predicted = logits.argmax(dim=1).detach().cpu().numpy()
            true_labels = batch.y.detach().cpu().numpy()
            graph_ids = batch.graph_id.detach().cpu().numpy() if hasattr(batch, 'graph_id') else np.arange(len(true_labels))

            for index, graph_id in enumerate(graph_ids):
                text_value = str(raw_df.iloc[int(graph_id)][text_col]) if int(graph_id) < len(raw_df) else ''
                rows.append({
                    'dataset': dataset,
                    'graph_id': int(graph_id),
                    'true_label': int(true_labels[index]),
                    'pred_label': int(predicted[index]),
                    'prob_fake': float(probabilities[index]),
                    'text': text_value,
                    'word_count': int(len(text_value.split())),
                })

    return pd.DataFrame(rows)


def load_or_build_predictions(prediction_path: str = '../results/predictions.csv') -> pd.DataFrame:
    prediction_file = Path(prediction_path)
    if prediction_file.exists():
        predictions = pd.read_csv(prediction_file)
    else:
        prediction_file.parent.mkdir(parents=True, exist_ok=True)
        frames = [generate_predictions_for_dataset(dataset) for dataset in ['politifact', 'gossipcop']]
        predictions = pd.concat(frames, ignore_index=True)
        predictions.to_csv(prediction_file, index=False)

    required_columns = {'true_label', 'pred_label', 'text'}
    missing_columns = required_columns - set(predictions.columns)
    if missing_columns:
        raise ValueError(f'Predictions file is missing required columns: {sorted(missing_columns)}')

    if 'dataset' not in predictions.columns:
        predictions['dataset'] = 'unknown'
    if 'prob_fake' not in predictions.columns:
        predictions['prob_fake'] = np.nan
    if 'word_count' not in predictions.columns:
        predictions['word_count'] = predictions['text'].astype(str).str.split().str.len().fillna(0).astype(int)

    predictions['true_label'] = predictions['true_label'].astype(int)
    predictions['pred_label'] = predictions['pred_label'].astype(int)
    predictions['is_error'] = predictions['true_label'] != predictions['pred_label']
    predictions['error_type'] = np.select(
        [
            (predictions['true_label'] == 0) & (predictions['pred_label'] == 1),
            (predictions['true_label'] == 1) & (predictions['pred_label'] == 0),
        ],
        ['false_positive', 'false_negative'],
        default='correct',
    )
    return predictions

## Load Predictions and Confusion Matrix

We first read the prediction table. If the expected CSV is missing, the notebook generates it from the trained HyDFake checkpoints so the analysis stays reproducible inside this workspace.

In [ ]:
predictions = load_or_build_predictions('../results/predictions.csv')

print(f'Total predictions: {len(predictions):,}')
display(predictions.head())

label_names = ['Real', 'Fake']
conf_matrix = confusion_matrix(predictions['true_label'], predictions['pred_label'])
report = classification_report(predictions['true_label'], predictions['pred_label'], target_names=label_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T

display(report_df.round(4))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
plt.tight_layout()
plt.show()

## False Positives, False Negatives, and Hard Samples

These sections inspect the actual mistakes. False positives are real articles predicted as fake, while false negatives are fake articles predicted as real. We also look for short, long, and ambiguous samples that tend to challenge the classifier.

In [ ]:
false_positives = predictions[(predictions['true_label'] == 0) & (predictions['pred_label'] == 1)].copy()
false_negatives = predictions[(predictions['true_label'] == 1) & (predictions['pred_label'] == 0)].copy()

print(f'False positives: {len(false_positives):,}')
print(f'False negatives: {len(false_negatives):,}')

display(false_positives[['dataset', 'word_count', 'prob_fake', 'text']].head(10))
display(false_negatives[['dataset', 'word_count', 'prob_fake', 'text']].head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=predictions, x='is_error', y='word_count', ax=axes[0], palette=['#2E86AB', '#E67E22'])
axes[0].set_title('Word Count vs Prediction Error')
axes[0].set_xlabel('Prediction error')
axes[0].set_ylabel('Word count')
axes[0].set_xticklabels(['Correct', 'Incorrect'])

sns.histplot(data=predictions, x='word_count', hue='is_error', bins=50, kde=True, stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Word Count Distribution by Error Flag')
axes[1].set_xlabel('Word count')
axes[1].set_ylabel('Density')
plt.tight_layout()
plt.show()

ambiguous_samples = predictions.assign(uncertainty=(predictions['prob_fake'] - 0.5).abs()).sort_values('uncertainty').head(10)
short_samples = predictions.sort_values('word_count').head(10)
long_samples = predictions.sort_values('word_count', ascending=False).head(10)

print('\nMost ambiguous samples')
display(ambiguous_samples[['dataset', 'true_label', 'pred_label', 'prob_fake', 'word_count', 'text']])

print('\nShortest samples')
display(short_samples[['dataset', 'true_label', 'pred_label', 'prob_fake', 'word_count', 'text']])

print('\nLongest samples')
display(long_samples[['dataset', 'true_label', 'pred_label', 'prob_fake', 'word_count', 'text']])

## Interpretation

Errors often arise from semantic ambiguity, sensational wording, and limited context. Short texts can omit enough evidence for the model to decide correctly, while very long texts may dilute the strongest cues.

False positives usually indicate that the model overreacted to fake-news style language in real articles. False negatives often mean that subtle manipulation was present but not captured by the embedding or graph representation. These failure modes are important when discussing the practical limits of the model in an academic report.